# NB_fig_tab_gen — Paper Assets Compilation
    
This notebook compiles the final `paper_assets/` package for the CXR Faithfulness manuscript.
**Scope:** CPU-only. Reads existing CSV/NPY/PNG files and generates canonical figures/tables.
**Does NOT** recompute attributions (IG, LIME, GradCAM) or load PyTorch models.


In [13]:
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/cxr_faithfulness')
else:
    # Local Drive shortcut (Windows) or repo root
    _candidates = [
        Path(r'G:/.shortcut-targets-by-id/1QLdpWtLkBeYlbYSTACFpC6Z05qNj3BCv/cxr_faithfulness'),
        Path(r'/path/to/cxr_faithfulness_local'),
        Path.cwd(),
    ]
    ROOT = next((p for p in _candidates if (p / 'results').exists()), _candidates[0])
print(f"ROOT set to {ROOT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT set to /content/drive/MyDrive/cxr_faithfulness


In [14]:
import os, shutil, json, cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 300
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

# Colorblind-safe palette
MODEL_COLORS = {'densenet121': '#01696f', 'convnextv2_tiny': '#da7101', 'swinb_lora': '#7a39bb'}
MODEL_LABELS = {'densenet121': 'DenseNet-121', 'convnextv2_tiny': 'ConvNeXtV2-Tiny', 'swinb_lora': 'Swin-B LoRA'}
print("Imports loaded.")


Imports loaded.


In [15]:
ASSETS_DIR = ROOT / 'paper_assets'
MAIN_FIG_DIR = ASSETS_DIR / 'main' / 'figures'
MAIN_TAB_DIR = ASSETS_DIR / 'main' / 'tables'
APP_FIG_DIR = ASSETS_DIR / 'appendix' / 'figures'
APP_TAB_DIR = ASSETS_DIR / 'appendix' / 'tables'

for d in [MAIN_FIG_DIR, MAIN_TAB_DIR, APP_FIG_DIR, APP_TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = ROOT / 'results'
FIGPATH = ROOT / 'figures'
IG_PATH = ROOT / 'ig_maps'
DATA_PATH = ROOT / 'data'
BBOX_PATH = DATA_PATH / 'processed' / 'consensus' / 'consensus_boxes_2of3.csv'
IMAGES_PATH = DATA_PATH / 'processed' / 'images'
IG_FN_PATH = IG_PATH / 'fn'

def class_slug(name: str) -> str:
    return str(name).replace('/', '_').replace(' ', '_')

def resolve_ig_path(p_str):
    return Path(str(p_str).replace('/content/drive/MyDrive/cxr_faithfulness', str(ROOT)))

print('Directory structure created.')


Directory structure created.


In [16]:
ASSET_REGISTRY = [
    {'id': 'Fig 01', 'file': 'main/figures/fig01_perclass_auc_heatmap.png', 'status': 'EXISTS', 'src': RESULTS_PATH / 'paper_figures' / 'fig1_perclass_auc_heatmap.png'},
    {'id': 'Fig 02', 'file': 'main/figures/fig02_macro_auc_ci_f1.png', 'status': 'EXISTS', 'src': RESULTS_PATH / 'paper_figures' / 'fig2_macro_auc_ci_f1.png'},
    {'id': 'Fig 03', 'file': 'main/figures/fig03_ig_faithfulness_gallery.png', 'status': 'GENERATE'},
    {'id': 'Fig 04', 'file': 'main/figures/fig04_fn_severity_stacked.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure_fn_severity_stacked.png'},
    {'id': 'Fig 05', 'file': 'main/figures/fig05_fp_wrongclass_or_gallery.png', 'status': 'EXISTS', 'src': RESULTS_PATH / 'paper_figures' / 'fig3_fp_wrongclass.png'},
    {'id': 'Fig 06', 'file': 'main/figures/fig06_miou_bar_chart.png', 'status': 'GENERATE'},
    {'id': 'Tab 01', 'file': 'main/tables/tab01_perclass_classification', 'status': 'EXISTS', 'src_csv': RESULTS_PATH / 'paper_tables' / 'table1_perclass_metrics.csv', 'src_tex': RESULTS_PATH / 'paper_tables' / 'table1_perclass_metrics.tex'},
    {'id': 'Tab 02', 'file': 'main/tables/tab02_model_classification_summary', 'status': 'EXISTS', 'src_csv': RESULTS_PATH / 'paper_tables' / 'table2_model_summary.csv', 'src_tex': RESULTS_PATH / 'paper_tables' / 'table2_model_summary.tex'},
    {'id': 'Tab 03', 'file': 'main/tables/tab03_faithfulness_summary', 'status': 'GENERATE'},
    {'id': 'Tab 04', 'file': 'main/tables/tab04_fn_bucket_by_severity', 'status': 'GENERATE'},
    {'id': 'Fig A01', 'file': 'appendix/figures/figA01_failure_gallery.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure4_failure_gallery.png'},
    {'id': 'Fig A02', 'file': 'appendix/figures/figA02_severity_barplot.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure5_severity_barplot.png'},
    {'id': 'Fig A03', 'file': 'appendix/figures/figA03_lime_ig_agreement.png', 'status': 'GENERATE'},
    {'id': 'Fig A04', 'file': 'appendix/figures/figA04_weight_rand_panel.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure_weight_rand_panel.png'},
    {'id': 'Fig A05', 'file': 'appendix/figures/figA05_weight_rand_cascade.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure_weight_rand_cascade.png'},
    {'id': 'Fig A06', 'file': 'appendix/figures/figA06_ablation_methods.png', 'status': 'EXISTS', 'src': FIGPATH / 'figure_ablation_methods.png'},
    {'id': 'Fig A07', 'file': 'appendix/figures/figA07_fn_qualitative_panel.png', 'status': 'GENERATE'},
    {'id': 'Fig A08', 'file': 'appendix/figures/figA08_entropy_calibration.png', 'status': 'EXISTS', 'src': FIGPATH / 'entropy_calibration.png'},
    {'id': 'Tab A01', 'file': 'appendix/tables/tabA01_faithfulness_per_pathology_full', 'status': 'GENERATE'},
    {'id': 'Tab A02', 'file': 'appendix/tables/tabA02_consensus_sensitivity', 'status': 'GENERATE'},
    {'id': 'Tab A03', 'file': 'appendix/tables/tabA03_ablation_method_comparison', 'status': 'DATA', 'src': RESULTS_PATH / 'ablation_method_comparison.csv'},
    {'id': 'Tab A04', 'file': 'appendix/tables/tabA04_weight_randomization_summary', 'status': 'DATA', 'src': RESULTS_PATH / 'weight_randomization_results.csv'},
    {'id': 'Tab A05', 'file': 'appendix/tables/tabA05_fn_sensitivity_tau', 'status': 'GENERATE'},
]


In [17]:
def df_to_latex(df, path, index=False):
    df_fmt = df.copy()
    for col in df_fmt.select_dtypes(include=['object']):
        df_fmt[col] = df_fmt[col].astype(str).str.replace('_', '\\_', regex=False).str.replace('%', '\\%', regex=False)
    col_format = 'l' + 'c'* (len(df_fmt.columns) - 1)
    latex_str = df_fmt.to_latex(index=index, escape=False, column_format=col_format)
    if '\\toprule' not in latex_str:
        latex_str = latex_str.replace('\\hline', '\\midrule', 1)
        latex_str = latex_str.replace('\\hline', '\\toprule', 1)
        latex_str = latex_str.replace('\\hline', '\\bottomrule')
    with open(path, 'w', encoding='utf-8') as f:
        f.write(latex_str)

def copy_asset(item):
    if 'src' in item:
        src = item['src']
        dst = ASSETS_DIR / item['file']
        if src.exists():
            shutil.copy2(src, dst)
            print(f"✅ Copied {item['id']}: {src.name}")
        else:
            print(f"⚠️ Missing source for {item['id']}: {src}")
    elif 'src_csv' in item:
        src_csv = item['src_csv']
        dst_csv = ASSETS_DIR / (item['file'] + '.csv')
        if src_csv.exists():
            shutil.copy2(src_csv, dst_csv)
        else:
            print(f"⚠️ Missing source CSV for {item['id']}: {src_csv}")

        src_tex = item['src_tex']
        dst_tex = ASSETS_DIR / (item['file'] + '.tex')
        if src_tex.exists():
            shutil.copy2(src_tex, dst_tex)
        else:
            print(f"⚠️ Missing source TEX for {item['id']}: {src_tex}")
        print(f"✅ Copied {item['id']} (.csv and/or .tex)")


In [18]:
print("--- COPY BLOCK ---")
for item in ASSET_REGISTRY:
    if item['status'] == 'EXISTS':
        copy_asset(item)
    elif item['status'] == 'DATA':
        src = item['src']
        dst_csv = ASSETS_DIR / (item['file'] + '.csv')
        dst_tex = ASSETS_DIR / (item['file'] + '.tex')
        if src.exists():
            shutil.copy2(src, dst_csv)
            df = pd.read_csv(src)
            df_to_latex(df, dst_tex)
            print(f"✅ Generated {item['id']} (.csv and .tex from DATA)")
        else:
            print(f"⚠️ Missing source data for {item['id']}: {src}")


--- COPY BLOCK ---
✅ Copied Fig 01: fig1_perclass_auc_heatmap.png
✅ Copied Fig 02: fig2_macro_auc_ci_f1.png
✅ Copied Fig 04: figure_fn_severity_stacked.png
✅ Copied Fig 05: fig3_fp_wrongclass.png
✅ Copied Tab 01 (.csv and/or .tex)
✅ Copied Tab 02 (.csv and/or .tex)
✅ Copied Fig A01: figure4_failure_gallery.png
✅ Copied Fig A02: figure5_severity_barplot.png
✅ Copied Fig A04: figure_weight_rand_panel.png
✅ Copied Fig A05: figure_weight_rand_cascade.png
✅ Copied Fig A06: figure_ablation_methods.png
✅ Copied Fig A08: entropy_calibration.png
✅ Generated Tab A03 (.csv and .tex from DATA)
✅ Generated Tab A04 (.csv and .tex from DATA)


In [19]:
print("--- GENERATE Tab 03 ---")
f_path = RESULTS_PATH / 'attribution_faithfulness_2of3.csv'
s_path = RESULTS_PATH / 'attribution_summary_by_model_2of3.csv'
if f_path.exists() and s_path.exists():
    df_f = pd.read_csv(f_path)
    df_s = pd.read_csv(s_path)
    # Get columns we need
    cols = ['model', 'pathology', 'n', 'miou_top50_mean', 'recall_top50_mean', 'precision_mass_mean']
    df_f_sub = df_f[[c for c in cols if c in df_f.columns]].copy()

    df_s_sub = df_s[['model', 'n', 'miou_top50_mean', 'recall_top50_mean', 'precision_mass_mean']].copy()
    df_s_sub.insert(1, 'pathology', 'ALL (Macro)')

    df_tab03 = pd.concat([df_f_sub, df_s_sub]).sort_values(['model', 'pathology']).reset_index(drop=True)
    df_tab03.columns = ['Model', 'Pathology', 'N', 'mIoU (top-50%)', 'Recall (top-50%)', 'Precision Mass']

    # Format floats
    for c in df_tab03.columns[3:]:
        df_tab03[c] = df_tab03[c].round(4)

    out_csv = ASSETS_DIR / 'main' / 'tables' / 'tab03_faithfulness_summary.csv'
    out_tex = ASSETS_DIR / 'main' / 'tables' / 'tab03_faithfulness_summary.tex'
    df_tab03.to_csv(out_csv, index=False)
    df_to_latex(df_tab03, out_tex)
    print("✅ Generated Tab 03")
else:
    print("⚠️ Missing data for Tab 03")


--- GENERATE Tab 03 ---
✅ Generated Tab 03


In [20]:
print("--- GENERATE Tab 04 ---")
fn_sev_path = RESULTS_PATH / 'fn_by_severity.csv'
if fn_sev_path.exists():
    df_sev = pd.read_csv(fn_sev_path)
    if 'pct_blind' in df_sev.columns:
        df_sev['pct_attentive'] = 1.0 - df_sev['pct_blind']
    elif 'n_attentive' in df_sev.columns and 'n_fn' in df_sev.columns:
        df_sev['pct_attentive'] = df_sev['n_attentive'] / df_sev['n_fn']

    df_tab04 = df_sev[['model', 'tier', 'n_fn', 'n_attentive', 'n_blind', 'pct_attentive']].copy()
    df_tab04.columns = ['Model', 'Severity Tier', 'N FNs', 'N Attentive', 'N Blind', '% Attentive']
    df_tab04['% Attentive'] = (df_tab04['% Attentive'] * 100).round(1).astype(str) + '%'

    out_csv = ASSETS_DIR / 'main' / 'tables' / 'tab04_fn_bucket_by_severity.csv'
    out_tex = ASSETS_DIR / 'main' / 'tables' / 'tab04_fn_bucket_by_severity.tex'
    df_tab04.to_csv(out_csv, index=False)
    df_to_latex(df_tab04, out_tex)
    print("✅ Generated Tab 04")
else:
    print("⚠️ Missing data for Tab 04")


--- GENERATE Tab 04 ---
✅ Generated Tab 04


In [21]:
print("--- GENERATE Fig 06 ---")
s_path = RESULTS_PATH / 'attribution_summary_by_model_2of3.csv'
if s_path.exists():
    df_s = pd.read_csv(s_path)
    fig, ax = plt.subplots(figsize=(6, 4))

    models = df_s['model'].values
    miou = df_s['miou_top50_mean'].values
    yerr = [
        miou - df_s['miou_top50_ci_lo'].values,
        df_s['miou_top50_ci_hi'].values - miou
    ]

    colors = [MODEL_COLORS.get(m, '#333') for m in models]
    labels = [MODEL_LABELS.get(m, m) for m in models]

    ax.bar(labels, miou, yerr=yerr, capsize=5, color=colors, edgecolor='black', alpha=0.8)
    ax.set_ylabel('Mean mIoU (top-50% mass vs 2-of-3 box)')
    ax.set_title('Attribution Faithfulness by Model')
    ax.grid(axis='y', alpha=0.3)

    out_fig = ASSETS_DIR / 'main' / 'figures' / 'fig06_miou_bar_chart.png'
    fig.savefig(out_fig, dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Generated Fig 06")
else:
    print("⚠️ Missing data for Fig 06")


--- GENERATE Fig 06 ---
✅ Generated Fig 06


In [22]:
print("--- GENERATE Fig 03 (IG Gallery) ---")
manifest_path = RESULTS_PATH / 'ig_manifest.csv'
if manifest_path.exists() and BBOX_PATH.exists():
    man_df = pd.read_csv(manifest_path)
    bbox_df = pd.read_csv(BBOX_PATH)

    man_patho = man_df[(man_df['subset'] == 'patho') & (man_df['target_class'].notna())]
    man_patho = man_patho[man_patho['image_id'].isin(bbox_df['image_id'])]

    # Select best model Swin-B LoRA
    df_swin = man_patho[man_patho['model'] == 'swinb_lora'].copy()

    # Pick 3 diverse pathologies: Cardiomegaly, Pleural effusion, Lung opacity
    target_pathologies = ['Cardiomegaly', 'Pleural effusion', 'Lung Opacity', 'Pulmonary fibrosis']
    selected_cases = []

    for patho in target_pathologies:
        df_p = df_swin[df_swin['target_class'] == patho]
        if not df_p.empty:
            df_p = df_p.sort_values('image_id')
            selected_cases.append(df_p.iloc[0])

    if selected_cases:
        fig, axes = plt.subplots(len(selected_cases), 3, figsize=(10, 3.5 * len(selected_cases)))
        if len(selected_cases) == 1:
            axes = [axes]

        for i, row in enumerate(selected_cases):
            img_id = str(row['image_id'])
            patho = row['target_class']

            img_file = IMAGES_PATH / f"{img_id}.png"
            img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE) if img_file.exists() else np.zeros((224,224))

            ig_file = resolve_ig_path(row['ig_path'])
            if ig_file.exists():
                ig_map = np.load(str(ig_file))
                if ig_map.ndim == 3: ig_map = np.abs(ig_map).sum(axis=0)
                else: ig_map = np.abs(ig_map)
                # normalize
                if ig_map.max() > 0: ig_map = ig_map / ig_map.max()
            else:
                ig_map = np.zeros((224, 224))

            boxes = bbox_df[(bbox_df['image_id'] == img_id) & (bbox_df['class_name'] == patho)]

            for j, ax in enumerate(axes[i]):
                ax.axis('off')
                ax.imshow(img, cmap='gray')
                if j == 1 or j == 2:
                    ax.imshow(ig_map, cmap='hot', alpha=0.45)
                if j == 2:
                    for _, box in boxes.iterrows():
                        rect = plt.Rectangle((box['x_min'], box['y_min']), box['x_max']-box['x_min'], box['y_max']-box['y_min'],
                                             edgecolor='red', facecolor='none', lw=2)
                        ax.add_patch(rect)

            axes[i, 0].set_title(f"{patho} - CXR", fontsize=10)
            axes[i, 1].set_title("IG Overlay", fontsize=10)
            axes[i, 2].set_title("IG + Consensus Bbox", fontsize=10)

        plt.tight_layout()
        out_fig = ASSETS_DIR / 'main' / 'figures' / 'fig03_ig_faithfulness_gallery.png'
        fig.savefig(out_fig, dpi=300, bbox_inches='tight')
        plt.close()
        print("✅ Generated Fig 03")
    else:
        print("⚠️ No valid cases found for Fig 03")
else:
    print("⚠️ Missing data for Fig 03")


--- GENERATE Fig 03 (IG Gallery) ---
✅ Generated Fig 03


In [23]:
print("--- GENERATE Fig A03 ---")
lime_summ_path = RESULTS_PATH / 'lime_agreement_summary.csv'
if lime_summ_path.exists():
    df_lime = pd.read_csv(lime_summ_path)

    piv = df_lime.pivot(index='target_class', columns='model', values='mean_lime_ig_iou')
    fig, ax = plt.subplots(figsize=(7, 6))

    # Ensure columns order if available
    cols_order = [m for m in ['densenet121', 'convnextv2_tiny', 'swinb_lora'] if m in piv.columns]
    piv = piv[cols_order]

    sns.heatmap(piv, annot=True, fmt=".3f", cmap='Blues', ax=ax, cbar_kws={'label': 'Mean LIME-IG mIoU'})
    ax.set_title("LIME vs. IG Top-50% Agreement")
    ax.set_ylabel("Pathology")
    ax.set_xlabel("Model")

    out_fig = ASSETS_DIR / 'appendix' / 'figures' / 'figA03_lime_ig_agreement.png'
    fig.savefig(out_fig, dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Generated Fig A03")
else:
    print("⚠️ Missing data for Fig A03")


--- GENERATE Fig A03 ---
✅ Generated Fig A03


In [24]:
print("--- GENERATE Fig A07 ---")
audit_path = RESULTS_PATH / 'fn_attribution_audit.csv'
if audit_path.exists() and BBOX_PATH.exists():
    audit_df = pd.read_csv(audit_path)
    bbox_df = pd.read_csv(BBOX_PATH)

    df_swin = audit_df[audit_df['model'] == 'swinb_lora'].copy()
    df_att = df_swin[df_swin['fn_bucket'] == 'attentive'].head(2)
    df_blind = df_swin[df_swin['fn_bucket'] == 'blind'].head(2)

    selected = pd.concat([df_att, df_blind])

    if not selected.empty:
        fig, axes = plt.subplots(len(selected), 3, figsize=(10, 3.5 * len(selected)))
        if len(selected) == 1:
            axes = [axes]

        for i, row in enumerate(selected.itertuples()):
            img_id = str(row.image_id)
            patho = row.missed_class

            img_file = IMAGES_PATH / f"{img_id}.png"
            img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE) if img_file.exists() else np.zeros((224,224))

            ig_file = IG_FN_PATH / f"{img_id}_{row.model}_{class_slug(patho)}_fn_ig.npy"
            if ig_file.exists():
                ig_map = np.load(str(ig_file))
                if ig_map.ndim == 3: ig_map = np.abs(ig_map).sum(axis=0)
                else: ig_map = np.abs(ig_map)
                if ig_map.max() > 0: ig_map = ig_map / ig_map.max()
            else:
                ig_map = np.zeros((224, 224))

            boxes = bbox_df[(bbox_df['image_id'] == img_id) & (bbox_df['class_name'] == patho)]

            for j, ax in enumerate(axes[i]):
                ax.axis('off')
                ax.imshow(img, cmap='gray')
                if j == 1 or j == 2:
                    ax.imshow(ig_map, cmap='hot', alpha=0.45)
                if j == 2:
                    for _, box in boxes.iterrows():
                        rect = plt.Rectangle((box['x_min'], box['y_min']), box['x_max']-box['x_min'], box['y_max']-box['y_min'],
                                             edgecolor='red', facecolor='none', lw=2)
                        ax.add_patch(rect)

            bucket_str = "Attentive FN" if row.fn_bucket == 'attentive' else "Blind FN"
            axes[i, 0].set_title(f"{patho} ({bucket_str})", fontsize=10)

        plt.tight_layout()
        out_fig = ASSETS_DIR / 'appendix' / 'figures' / 'figA07_fn_qualitative_panel.png'
        fig.savefig(out_fig, dpi=300, bbox_inches='tight')
        plt.close()
        print("✅ Generated Fig A07")
    else:
        print("⚠️ No Swin FN cases found for Fig A07")
else:
    print("⚠️ Missing data for Fig A07")


--- GENERATE Fig A07 ---
✅ Generated Fig A07


In [25]:
print("--- GENERATE Appendix Tables ---")
# Tab A01
f_path = RESULTS_PATH / 'faithfulness_results.csv'
if f_path.exists():
    df_f = pd.read_csv(f_path)
    out_csv = ASSETS_DIR / 'appendix' / 'tables' / 'tabA01_faithfulness_per_pathology_full.csv'
    out_tex = ASSETS_DIR / 'appendix' / 'tables' / 'tabA01_faithfulness_per_pathology_full.tex'
    df_f.to_csv(out_csv, index=False)
    df_to_latex(df_f, out_tex)
    print("✅ Generated Tab A01")

# Tab A02
f2_path = RESULTS_PATH / 'attribution_summary_by_model_2of3.csv'
f3_path = RESULTS_PATH / 'attribution_summary_by_model_3of3.csv'
if f2_path.exists() and f3_path.exists():
    df2 = pd.read_csv(f2_path)
    df3 = pd.read_csv(f3_path)
    df2['consensus'] = '2-of-3'
    df3['consensus'] = '3-of-3'
    df_comb = pd.concat([df2, df3])
    out_csv = ASSETS_DIR / 'appendix' / 'tables' / 'tabA02_consensus_sensitivity.csv'
    out_tex = ASSETS_DIR / 'appendix' / 'tables' / 'tabA02_consensus_sensitivity.tex'
    df_comb.to_csv(out_csv, index=False)
    df_to_latex(df_comb, out_tex)
    print("✅ Generated Tab A02")

# Tab A05
fn_sens = RESULTS_PATH / 'fn_sensitivity_tau.csv'
if fn_sens.exists():
    df_sens = pd.read_csv(fn_sens)
    out_csv = ASSETS_DIR / 'appendix' / 'tables' / 'tabA05_fn_sensitivity_tau.csv'
    out_tex = ASSETS_DIR / 'appendix' / 'tables' / 'tabA05_fn_sensitivity_tau.tex'
    df_sens.to_csv(out_csv, index=False)
    df_to_latex(df_sens, out_tex)
    print("✅ Generated Tab A05")
else:
    print("⚠️ Missing data for Tab A05")


--- GENERATE Appendix Tables ---
✅ Generated Tab A01
✅ Generated Tab A02
✅ Generated Tab A05


In [26]:
print("--- MANIFEST ---")
manifest_data = []
readme_lines = [
    "# Paper Assets — cxr_faithfulness",
    "",
    "Canonical figures and tables for manuscript. Generated by `notebooks/NB_fig_tab_gen.ipynb`.",
    "",
    "## Main text",
    "| ID | File |",
    "|----|------|"
]

for item in ASSET_REGISTRY:
    p = ASSETS_DIR / item['file']
    # Check if exists
    exists = False
    size = 0
    if p.exists():
        exists = True
        size = p.stat().st_size
    elif (ASSETS_DIR / (item['file'] + '.csv')).exists():
        exists = True
        p = ASSETS_DIR / (item['file'] + '.csv')
        size = p.stat().st_size

    manifest_data.append({
        'id': item['id'],
        'path': item['file'],
        'status': 'OK' if exists else 'MISSING',
        'bytes': size
    })

    if item['id'].startswith('Fig 0') or item['id'].startswith('Tab 0'):
        readme_lines.append(f"| {item['id']} | {item['file']} |")

readme_lines.extend(["", "## Appendix", "| ID | File |", "|----|------|"])

for item in ASSET_REGISTRY:
    if item['id'].startswith('Fig A') or item['id'].startswith('Tab A'):
        readme_lines.append(f"| {item['id']} | {item['file']} |")

readme_lines.extend([
    "",
    "## Regenerate",
    "Run all cells in `NB_fig_tab_gen.ipynb` (CPU only)."
])

manifest_df = pd.DataFrame(manifest_data)
manifest_df.to_csv(ASSETS_DIR / 'manifest.csv', index=False)

with open(ASSETS_DIR / 'README.md', 'w') as f:
    f.write('\n'.join(readme_lines))

print("✅ manifest.csv and README.md written.")


--- MANIFEST ---
✅ manifest.csv and README.md written.


In [27]:
print("--- VERIFICATION CHECKLIST ---")
all_ok = True
for item in ASSET_REGISTRY:
    p = ASSETS_DIR / item['file']
    if 'figures' in item['file']:
        if not p.exists() or p.stat().st_size < 10000:
            print(f"[FAIL] {item['id']} image missing or too small")
            all_ok = False
    else:
        # Table
        p_csv = ASSETS_DIR / (item['file'] + '.csv')
        p_tex = ASSETS_DIR / (item['file'] + '.tex')
        if not p_csv.exists() or not p_tex.exists():
            print(f"[FAIL] {item['id']} table missing .csv or .tex")
            all_ok = False

if all_ok:
    print("✅ PAPER ASSETS COMPLETE")
else:
    print("⚠️ SOME ASSETS MISSING OR INVALID")


--- VERIFICATION CHECKLIST ---
✅ PAPER ASSETS COMPLETE
